In [1]:
from pathlib import Path
from src.data.pipeline import run_data_pipeline
import pandas as pd
import numpy as np

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Data file exists:", data_path.exists())

Data file exists: True


### Step 1: Features and target

In [2]:
# Load processed dataset
df = run_data_pipeline(
    "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)
print(df.shape)

(7043, 21)


In [3]:
# Target variables are Yes and No, both stored as strings
# Convert strings into binary values for consistency
y = df["Churn"].map({"No": 0, "Yes": 1})

In [4]:
# Feature matrix
X = df.drop(columns=["customerID", "Churn"])

print(X.shape)
print(y.shape)
print(y.value_counts(normalize=True))

(7043, 19)
(7043,)
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


### Step 2: Train/test split
Divide the dataset into two groups
- Training set (80%): Teaches the model patterns associated with churn.
- Test set (20%): Evaluates how well the model performs on customers it hasn't seen during training.

Because approximately 26.5% of customers churn, both groups should preserve that distribution.\
This is called a **stratified split**.

The function takes several arguments:
| Parameter | Purpose | My choice |
|:----------|:--------|:----------|
| x | Feature matrix | 19 predictor columns |
| y | Target variable | Binary churn outcome |
| test_size | Fraction reserved for testing | 0.20 |
| random_state | Makes the split reproducible | 42 |
| stratify | Preserves the target distribution | y |

The function returns four objects:\
**X_train, X_test, y_train, y_test**

In [5]:
# Import train_test_split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
    )

In [6]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training churn rate:", y_train.mean())
print("Testing churn rate:", y_test.mean())

Training features: (5634, 19)
Testing features: (1409, 19)
Training churn rate: 0.2653532126375577
Testing churn rate: 0.2654364797728886


### Step 3: Preprocessing pipeline

Logistic regression cannot directly process all 19 columns in thier current form.\
\
There are two main types of features:
- **Numerical**: tenure, MonthlyCharges, TotalCharges
- **Categorical**: Contract, InternetService, PaymentMethod, gender, etc.
\
These need to be transformed differently.

In [7]:
# Identify numerical columns
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [8]:
# Identify categorical columns
categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

In [9]:
print("Numerical:", len(numeric_features))
print("Categorical:", len(categorical_features))
print(categorical_features)

Numerical: 3
Categorical: 16
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [10]:
# Transformers

# StandardScaler transforms numerical values based on their training-set mean and standard deviation. 
# OneHotEncoder converts categories into indicator columns. Use handle_unkown="ignore", it allows the model to preprocess a category at prediction time that wasn't encountered during training, encoding it as zeros for that feature's category indicators.
from sklearn.preprocessing import StandardScaler, OneHotEncoder 

#
from sklearn.compose import ColumnTransformer 

In [11]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

### Step 4: Machine Learning Pipeline

Connect the preprocessing transformer to a logistic regression model. \
The goal is to create a single object that handles both feature preparation and model training.

In [12]:
# Import classes
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [13]:
# Create pipepline
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

In [14]:
# Train model
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

### Step 5: Evaluate Model


This model can produce two types of predictions:
1. **Churn prediction**: Returns either 0 (retained) or 1 (churned), using the default classification threshold of 0.5.
2. **Churn probability**: Returns a decimal which represents the estimated chance of churn, subject to how well its probabilities are calibrated.

In [15]:
# Churn prediction
y_pred = model_pipeline.predict(X_test)

# Churn probability
y_proba = model_pipeline.predict_proba(X_test)[:, 1] # [:, 1] selects the probability of class 1, which represents churn

#### Evaluation Metrics

| Metric | What it tells us|
|:-------|:----------------|
| Accuracy | Percentage of all predictions that were correct |
| Precision | Of customers predicted to churn, how many actually churned? |
| Recall | Of customers who actually churned, how many did we identify? |
| F1 score | Balance between precision and recall |
| ROC-AUC | How well the model ranks churners above non-churners |
| PR-AUC (average precision) | Precision-recall performance across thresholds |

In [16]:
# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [17]:
# Calculate metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred))
print("ROC-AUC", roc_auc_score(y_test, y_proba))
print("Average Precision", average_precision_score(y_test, y_proba))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.8055358410220014
Precision: 0.6572327044025157
Recall: 0.5588235294117647
F1 score: 0.6040462427745664
ROC-AUC 0.8420341522643313
Average Precision 0.6337024099769349
[[926 109]
 [165 209]]


In [18]:
# Create a validation set using existing training data
X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    train_size=0.80,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Development:", X_dev.shape)
print("Validation:", X_val.shape)
print("Development churn rate:", y_dev.mean())
print("Validation churn rate:", y_val.mean())

Development: (4507, 19)
Validation: (1127, 19)
Development churn rate: 0.2653649877967606
Validation churn rate: 0.2653061224489796


##### Updated Split Data
| Dataset | Customers | Churn rate | Purpose |
|:--------|:----------|:-----------|:--------|
| Development | 4,507 | 26.54% | Train and improve model |
| Validation | 1,127 | 26.53% | Compare thresholds and model configurations |
| Test | 1,409 | 26.54% | Final evaluation |

#### Results

##### Baseline Model
- This model achieves approximately 80.6% accuracy compared with 73.5% for a model that always predicts retention. Which is an improvement of about 7.1%.
- The ROC-AUC of 0.842 is promising, suggesting the model ranks churners above non-churners reasonably well.
- Its average precision of 0.634 also substantially exceeds the churn prevalance of approximately 0.265.

##### Confusion Matrix

- Of the 374 customers who actually churned, the model identified 209 and missed 165. This means **44.1% of churners went undectected at the default threshold**.
- For a retention business, this matters because those missed customers represent potential lost revenue and missed intervention opportunities.
- Lowering the threshold will create false positives, so the goal is to understand that tradeoff before deciding what to change.

##### Improving the Classification Threshold
Currently, logistic regression uses a default threshold of 0.50. \
That means:
- Predicted churn probability of 0.49 → predict retention.
- Predicted churn probability of 0.51 → predict churn.
There's nothing inherently optimal about 0.50 for customer retention. \
Lowering the threshold flags more customers. It generally increases recall, but can reduce precision. \
The goal is to choose a threshold based on the economics of a retention campaigns, not simply whichever produces the highest accuracy.

### Step 6: Evaluate Classification Thresholds

**Can more customers be identified as at risk of churning by changing the classification threshold?**

As the threshold lowers, typically:
- **Recall increase**: We identifiy more customers who actually churn.
- **Precision decrease**: More customers are incorrectly flagged.
- **Number of customers flagged increase**: Retention campaigns become more expensive.

For this project, the threshold that produces the greatest expected business value under realistic campaign costs and retention effectiveness assumptions, is the goal.

In [19]:
# Train a fresh model using the development customers
# Use clone() to create a new, unfitted copy of the existing pipeline
from sklearn.base import clone

dev_model = clone(model_pipeline)
dev_model.fit(X_dev, y_dev)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [20]:
# Generate validation probabilities
val_proba = dev_model.predict_proba(X_val)[:, 1] #Using predict_proba() to control the classification threshold

In [21]:
# Experiment w/ threshold & evaluate
thresholds = [0.30, 0.40, 0.50]

for threshold in thresholds:
    val_pred = (val_proba >= threshold).astype(int)

    print("Precision:", precision_score(y_val, val_pred))
    print("Recall:", recall_score(y_val, val_pred))
    print("F1:", f1_score(y_val, val_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_val, val_pred))

Precision: 0.5432692307692307
Recall: 0.7558528428093646
F1: 0.6321678321678321
Confusion matrix:
[[638 190]
 [ 73 226]]
Precision: 0.6172839506172839
Recall: 0.6688963210702341
F1: 0.6420545746388443
Confusion matrix:
[[704 124]
 [ 99 200]]
Precision: 0.6965811965811965
Recall: 0.5451505016722408
F1: 0.6116322701688556
Confusion matrix:
[[757  71]
 [136 163]]


##### Dev_model Results w/ Different Classification Thresholds

| Metric | 0.30 | 0.40 | 0.50 |
|:-------|-----:|-----:|----:|
| Precision | 54.33% | 61.73% | 69.66% |
| Recall | 75.59% | 66.89% | 54.52% |
| F1 score | 0.632 | 0.642 | 0.612 |
| True positives | 226 | 200 | 163 |
| False positives | 190 | 124 | 71 |
| False negative | 73 | 99 | 136 |
| Customers flagged | 416 | 324 | 234 |


For now, **0.40 is a reasonable F1-based benchmark**, but it isn't the optimal retention threshold because:
- **0.30**: Best for identifying more churners, but generates the most alerts.
- **0.40**: Highest F1 score of the three, balancing precision and recall.
- **0.50**: Most selective, producing fewer unnecessary alerts but missing more churners.

The best business decision depends on the cost of contacting customers, their potential value, and whether intervention would actually prevent churn.

#### First Retention Economics Model

Given a limited retention budget, which customers should we prioritiz to maximize expected financial benefit?

**_Expected Net Value<sub>i</sub> = P(churn<sub>i</sub>) x V<sub>i</sub> x E<sub>i</sub> - C<sub>i</sub>_**

Where:
| Variable | Meaning |
|:---------|:--------|
| P(churn<sub>i</sub>) | Predicted probability that customer i churns |
| V<sub>i</sub> | Estimated financial value preserved if the customer is retained |
| E<sub>i</sub> | Assumed probability an intervention prevents churn, conditional on otherwise churning |
| C<sub>i</sub> | Cost of contacting or incentivizing that customer |

This is a simplified decision model. The dataset doesn't measure intervention effectiveness, so that quantity will clearly be labeled as an assumption rather than a learned casual effect.

For the first prototype, assume:
- A 12-month retention horizon.
- A 70% contribution margin.
- A $20 intervention cost per contacted customer.
- A 20% assumed intervention effectiveness.

These are **illustrative assumptions**, not measured business facts.

We'll estimate the value of retaining a customer as: \
**_V<sub>i</sub> = MonthlyCharges<sub>i</sub> x 12 x 0.70_**

In [22]:
# Create a retention analysis DataFrame
retention_df = X_val[["MonthlyCharges"]].copy()
retention_df["churn_probability"] = val_proba

In [23]:
# Calculate customer value
retention_df["estimated_customer_value"] = retention_df["MonthlyCharges"] * 12 * 0.70

In [24]:
# Calculate expected net value
intervention_cost = 20
intervention_effectiveness = 0.20
retention_df["expected_net_value"] = retention_df["churn_probability"] * retention_df["estimated_customer_value"] * intervention_effectiveness - intervention_cost

In [25]:
print(retention_df.head())
print(retention_df["expected_net_value"].describe())

      MonthlyCharges  churn_probability  estimated_customer_value  \
6167           90.05           0.196074                    756.42   
740            65.00           0.018649                    546.00   
1399           19.55           0.052969                    164.22   
4392           19.20           0.166494                    161.28   
6950           49.40           0.254350                    414.96   

      expected_net_value  
6167            9.662864  
740           -17.963505  
1399          -18.260289  
4392          -14.629569  
6950            1.109039  
count    1127.000000
mean       13.198249
std        35.741841
min       -19.944964
25%       -16.467986
50%        -1.245373
75%        38.873233
max       117.095659
Name: expected_net_value, dtype: float64


The most interesting finding is that the median expected net value is negative, even though the mean is positive. \
This suggests that a relatively smaller group of high-value opportunities is pulling the average upward. \
In other words, **contacting everyone is unlikely to be the most efficient use of a limited retention budget**.

##### Compare two retention strategies

Compare two ways a business might select customers for a retention campaign. \
**Strategy A – Highest churn risk:** Contact customers with the highest predicted probability of churning. \
**Strategy B – Highest expected net value:** COntact customers with the greatest estimated financial benefit from an intervention.\
\
Campaign size – 100 customers \
Intervention cost – $20 per customer \
**Total campaign budget** – **$2,000**

In [26]:
# Select customers by churn probabilty
risk_strategy = pd.DataFrame(retention_df.nlargest(100, "churn_probability"))

In [27]:
# Select customers by expected net value
value_strategy = pd.DataFrame(retention_df.nlargest(100, "expected_net_value"))

In [28]:
# Calculate campaign outcomes
# Risk strategy
risk_net_value = risk_strategy["expected_net_value"].sum()
risk_churn_probability = risk_strategy["churn_probability"].mean()
risk_customer_value = risk_strategy["estimated_customer_value"].mean()

# Value strategy
value_net_value = value_strategy["expected_net_value"].sum()
value_churn_probability = value_strategy["churn_probability"].mean()
value_customer_value = value_strategy["estimated_customer_value"].mean()

In [29]:
print(risk_net_value)
print(risk_churn_probability)
print(risk_customer_value)

8443.962318597478
0.7212451955103742
722.6226000000001


In [30]:
print(value_net_value)
print(value_churn_probability)
print(value_customer_value)

8904.375766568024
0.6972814274340332
784.5726


##### Compare the results

| Metric | Highest churn risk | Highest expected value |
|:-------|:-------------------|:-----------------------|
| Customers contacted | 100 | 100 |
| Campaign cost | $2,000 | $2,000 |
| Total expected net value | $8,443.96 | $8,904.38 |
| Average churn probability | 72.12% | 69.73% |
| Average customer value | $722.62 | $784.57 |

The value-based strategy targets customers with slightly lower average churn probability (69.73% vs 72.12%), but higher estimated financial value ($784.57 vs $722.62). \
That tradeoff produces an additional $460.41 in modeled expected net value. \
This is precisely the distinction between a **prediction system** and a **decision-support system**.

One qualification: because we're selecting the highest expected-value customers using the same formula we use to evaluate them, an improvement in modeled value is expected. The important next question is whether the strategy remains attractive under different assumptions.

#### Sensitivity Analysis

The current model assumes that retention interventions ae 20% effective. \
\
But what if the true effectiveness is only 5%? What if it's 30% \
\
We need to test whether our reccomendations are robust to changes in this assumption.\
\
At low intervention effectiveness, both campaigns could lose money. This is why a retention reccomendation system should consider uncertainty, not just model predictions.

In [31]:
# Sensitivity analysis
effectiveness_rates = [0.05, 0.10, 0.20, 0.30]

In [32]:
results = []
for rate in effectiveness_rates:
    scenario_df = retention_df.copy()
    scenario_df["expected_net_value"] = scenario_df["churn_probability"] * scenario_df["estimated_customer_value"] * rate - intervention_cost
    value_strategy = scenario_df.nlargest(100, "expected_net_value")
    value_total_net_value = float(value_strategy["expected_net_value"].sum())

    results.append(
        {
            "effectiveness_rate": rate,
            "expected_net_value": value_total_net_value
        }
    )

sensitivity_df = pd.DataFrame(results)

In [33]:
print(results)
print(sensitivity_df.head())

[{'effectiveness_rate': 0.05, 'expected_net_value': 726.0939416420057}, {'effectiveness_rate': 0.1, 'expected_net_value': 3452.1878832840116}, {'effectiveness_rate': 0.2, 'expected_net_value': 8904.375766568024}, {'effectiveness_rate': 0.3, 'expected_net_value': 14356.563649852034}]
   effectiveness_rate  expected_net_value
0                0.05          726.093942
1                0.10         3452.187883
2                0.20         8904.375767
3                0.30        14356.563650


The expected net value increases by the same amount each time effectiveness rises by 10 percentage points:
\
\
**Approximately $5,452.19 per 10-perecentage-point increase.**
\
\
This happens because the current formula is linear in intervention effectiveness. \
\
Furthermore, since we're applying the same effectiveness rate and intervention cost to every customer, changing effectiveness does **not change the ranking of customers** when the rate is positive. \
\
This is an important limitation of the current model.

##### Budget-constrained retention campaigns

So far, we've always selected exactly 100 customers. \
But real businesses generally work with budgets rather than fixed customer counts. \
For example, a retention team might have $1,000, $2,000, or $5,000 available for outreach.
\
The goal is to build a function that answers: \
**Given a campaign budget, which customers should we contact to maximize net value?**

In [34]:
# Create a function
def select_retention_customer(df, budget, intervention_cost):
    # Validate the budget and intervention cost
    if budget < 0:
        raise ValueError("Invalid budget.")
    if intervention_cost <= 0:
        raise ValueError("Invalid intervention cost.")
    # Calculate num of customers budget can cover
    num_customers = budget // intervention_cost
    # Exclude customers whose expected net value is zero or negative
    filtered_df = df[df["expected_net_value"] > 0]
    # Select the highest-value customers up to the budget limit
    selected_customers = filtered_df.nlargest(int(num_customers), "expected_net_value")
    # Return a DataFrame containing those customers
    return selected_customers


In [35]:
# test function
selected_customers = select_retention_customer(
    retention_df,
    budget=2000,
    intervention_cost=20
)

print("Customers selected:", len(selected_customers))
print("Campaign cost:", len(selected_customers) * 20)
print("Expected net value:", selected_customers["expected_net_value"].sum())

Customers selected: 100
Campaign cost: 2000
Expected net value: 8904.375766568024


In [36]:

df = pd.DataFrame(
    {
        "expected_net_value": [50, 20, -10, 80, 0]
    }
)

selected = select_retention_customer(
    df,
    budget=40,
    intervention_cost=20
)

selected_value_list = selected["expected_net_value"].tolist()
print(selected_value_list)
# print(selected.nlargest(2, selected_value_list)) 

[80, 50]


## Model explainability

**The model predicts customer churn, but can you explain why?**

#### Logistic regression coefficients

The logisitic regression model assigns a coefficient to every feature after preprocessing.\
\
The coefficients tell us how each feature contributes to the model's predicted log-odds of churn.
- **Positive coefficient:** Increases predicted churn log-odds, holding other features constant.
- **Negative coefficient:** Decreases predicted churn log-odds, holding other features constant.
- **Coefficient magnitude:** Indicates the strength of the relationship on the transformed feature's scale.

In [37]:
# Retrieve the trained preprocessing step
fitted_preprocessor = dev_model.named_steps["preprocessor"]

In [38]:
# Retrieve the transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

In [39]:
# Retrieve the logistic regression coefficients
coefficients = dev_model.named_steps["classifier"].coef_[0]

In [40]:
print("Number of features:", len(feature_names))
print("Number of coefficients:", len(coefficients))

Number of features: 46
Number of coefficients: 46


#### Build a coefficient analysis DataFrame

In [41]:
# Create the DataFrame
coefficient_df = pd.DataFrame(
    {
    "feature": feature_names,
    "coefficient": coefficients
    }
)

In [42]:
# Find the strongest associations
desc_coefficients = coefficient_df.sort_values(by="coefficient", ascending=False)
asec_coefficients = coefficient_df.sort_values(by="coefficient", ascending=True)
largest_10_coeff = desc_coefficients.head(10)
most_neg_10_coeff = asec_coefficients.head(10)
print(largest_10_coeff)
print(most_neg_10_coeff)

                                feature  coefficient
37         cat__Contract_Month-to-month     0.650936
17     cat__InternetService_Fiber optic     0.608932
2                     num__TotalCharges     0.501831
33                 cat__StreamingTV_Yes     0.207002
36             cat__StreamingMovies_Yes     0.176875
19               cat__OnlineSecurity_No     0.165563
28                  cat__TechSupport_No     0.152314
44  cat__PaymentMethod_Electronic check     0.148594
15               cat__MultipleLines_Yes     0.109072
22                 cat__OnlineBackup_No     0.047090
                                      feature  coefficient
0                                 num__tenure    -1.213312
39                     cat__Contract_Two year    -0.858246
16                   cat__InternetService_DSL    -0.620178
1                         num__MonthlyCharges    -0.486273
13                      cat__MultipleLines_No    -0.295975
40                   cat__PaperlessBilling_No    -0.284334
20  

Something unusual to note:
- **TotalCharges** has a positive coefficient of +0.502.
- **MonthlyCharges** has a negative coefficient of -0.486.
- **tenure** has a strongly negative coefficient of -1.213.
At first glance, it may seem as though higher monthly charges reduce churn while higher total charges increase it.\
That interpretation is misleading.\
**TotalCharges** is closely related to both **tenure** and **MonthlyCharges**. Including all three creates correlated predictors, which can make individual logisitic regression coefficients difficult to interpret.\
\
There's another issue: **OneHotEncoder** includes every category for categorical variables. Combined with the intercept, this creates redundant representations, so individual category coefficients are not uniquely interpretable without considering the encoding and regularization.\
**These features shouldn't be removed immediately**. Instead, use an additional explanation technique that evaluates features through their impact on predictions.

#### Permutation Importance
Permutation importance helps answer:\
**How much does model performance deteriorate when we randomly shuffle the values of a particular feature?** \
Unlike examining transformed logisitic regression coefficients, permutation importance can be calculated using the original 19 input features.\
This makes the results easier to communicate to business stakeholders.

In [43]:
# import permutation importance
from sklearn.inspection import permutation_importance


In [44]:
# calculate importance on the validation set
perm_result = permutation_importance(
    dev_model,
    X_val,
    y_val,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42
)

This measures how much validation ROC-AUC decreases when each original feature is shuffled.\
Using the validation set rather than the test set to avoid introducing another development decision based on test performance.

In [45]:
# create the importance DataFrame
importance_df = pd.DataFrame(
    {
        "feature": X_val.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std
    }
)

In [46]:
# sort the importance DataFrame by mean
sorted_importance_df = importance_df.sort_values(by="importance_mean", ascending=False)
print(sorted_importance_df.head(10))

             feature  importance_mean  importance_std
4             tenure         0.161467        0.012447
14          Contract         0.043919        0.006655
7    InternetService         0.042594        0.007052
17    MonthlyCharges         0.017882        0.002431
18      TotalCharges         0.016620        0.003195
16     PaymentMethod         0.006362        0.000950
8     OnlineSecurity         0.006277        0.001063
12       StreamingTV         0.005429        0.001542
15  PaperlessBilling         0.005346        0.001702
13   StreamingMovies         0.005290        0.000768


##### Key Findings
- **Tenure dominates.** Shuffling tenure decreases ROC-AUC by approximately 0.161, much more than any other individual feature. The model relies heavily on how long a customer has been with the company.
- **Contract type and internet service  are the next strongest features.** Their importance scores ae very similar, approximately 0.044 and 0.043.
- **MonthlyCharges and TotalCharges contribute additional predictive information**, although their correlation with tenure and each other complicates interpretation.
These findings are consistent with my earlier exploratory analysis, but they don't establish that changing a customer's contract or service would reduce churn.

Permutation importance measures how much the **fitted model** relies on a feature, not how inherently important that feature is in the real world.\
Because **tenure** and **TotalCharges** are related, shuffling one feature may leave some of its information available through the other.\
Also, my five permutation repates provide and initial estimate, but they're not enough to establish that small differences between features are meaningful.

In [47]:
# save permutation importance as csv
output_dir = project_root / "reports" / "model_explainability"
output_dir.mkdir(parents=True, exist_ok=True)

sorted_importance_df.to_csv(output_dir / "permutation_importance.csv", index=False)


In [48]:
# save logisitic regression coefficients as csv
coefficient_df = coefficient_df.sort_values(
    by="coefficient",
    ascending=False
)
coefficient_df.to_csv(output_dir / "logistic_regression_coefficients.csv", index=False)

In [49]:
print("Permutation importance saved:",
      (output_dir / "permutation_importance.csv").exists())

print("Coefficients saved:",
      (output_dir / "logistic_regression_coefficients.csv").exists())

Permutation importance saved: True
Coefficients saved: True


## Model Persistence and Inference

The goal here is to make predictions without retraining the model every time we open the notebook.\
\
I'll use **joblib** to save the entire scikit-learn pipeline, including its fitted preprocessing steps and logisitic regression classifier.

**Which model should be saved?**\
\
There are currently two trained pipeline:
|Pipeline|Training data|Purpose|
|:-------|:------------|:------|
|dev_model|4,507 development customers| Validaiton experiments and explainability|
|model_pipeline| 5,634 training customers|Baseline model trained on the full original training split|

For the reusable model artifact, I'll use **model_pipeline**, since it has trained on more data.\
\
One important qualification: I explored thresholds on the validation set after initially fitting this model.\
I'll treat this artifact **as a development prototype**, not a fully independent validated production model.

In [50]:
# import joblib
import joblib

In [51]:
# create model directory
model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

In [52]:
# save the model
model_path = model_dir / "churn_logistic_regression.joblib"
joblib.dump(model_pipeline, model_path)
print("Model size (KB):", round(model_path.stat().st_size / 1024, 2))

Model size (KB): 7.38


#### Verify model persistence

In [53]:
# load the saved model
loaded_model = joblib.load(model_path)

In [54]:
# compare predictions
sample_customers = X_test.head(5)

original_predictions = model_pipeline.predict(sample_customers)
loaded_predictions = loaded_model.predict(sample_customers)

original_predictions_proba = model_pipeline.predict_proba(sample_customers)
loaded_predictions_proba = loaded_model.predict_proba(sample_customers)

assert np.array_equal(original_predictions, loaded_predictions)
assert np.allclose(original_predictions_proba, loaded_predictions_proba)

#### Build a reusable prediction module

In [55]:
# test predict.py
from src.models.predict import load_churn_model, predict_churn

model = load_churn_model(model_path)

# create a realistic input
sample_customers = X_test.head(5)
reordered_customers = sample_customers[sample_customers.columns[::-1]].copy()
reordered_customers["extracCustomerID"] = [1, 2, 3, 4, 5]

probabilities = predict_churn(model, reordered_customers)
og_probabilities = model_pipeline.predict_proba(sample_customers)[:, 1]

assert np.allclose(probabilities, og_probabilities)

print("Number of predictions:", len(probabilities))
print(probabilities)
print(og_probabilities)

Number of predictions: 5
[0.04605857 0.68364788 0.05872542 0.40112026 0.02163739]
[0.04605857 0.68364788 0.05872542 0.40112026 0.02163739]


## Connect Predictions to Retention Recommendations

Now I'll combine the two major components of the Customer Retention Intelligence Platform:
- src/model/predict.py – estimates churn probability.
- src/retention/recommend.py – selects customers based on expected net value and campaign budget.
\
\
The missing piece is converting model predictions into the economic inputs required by the recommendation engine.

In [56]:
# test score_retention_customers
from src.retention.recommend import (
    score_retention_customers,
    select_retention_customer,
    generate_retention_campaign
)

scored_customers = score_retention_customers(
    model,
    X_test,
    contribution_margin=0.70,
    intervention_effectiveness=0.20,
    intervention_cost=20
)

In [57]:
selected_customers = select_retention_customer(scored_customers, budget=2000, intervention_cost=20)
print(len(selected_customers))
print(len(selected_customers) * intervention_cost)
print(selected_customers["expected_net_value"].sum())
print(selected_customers["churn_probability"].mean())

100
2000
9203.721089603076
0.7164818036350313


In [58]:
campaign_customers = generate_retention_campaign(model=model, customers=X_test, budget=2000)
print(len(campaign_customers))
print(len(campaign_customers) * intervention_cost)
print(campaign_customers["expected_net_value"].sum())
print(campaign_customers["churn_probability"].mean())

100
2000
9203.721089603076
0.7164818036350313
